In [1]:
%load_ext sql
%sql sqlite:///characters.db

In [2]:
%%sql

CREATE TABLE IF NOT EXISTS species (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    species      TEXT    NOT NULL UNIQUE,
    description TEXT    NOT NULL,
    rarity TEXT NOT NULL DEFAULT 'Common'
);

CREATE TABLE IF NOT EXISTS species_buffs (
    species_id      INTEGER NOT NULL REFERENCES species(id),
    attribute       TEXT    NOT NULL REFERENCES attributes(attribute),
    level_modifier  INTEGER NOT NULL,  
    PRIMARY KEY (species_id, attribute)
);


 * sqlite:///characters.db
Done.
Done.


[]

In [3]:
import sqlite3

SPECIES = [
    ('Human', 'Balanced baseline species with no innate buffs or debuffs.', "Common"),
    ('Elf',   'Ancient race with unmatched speed and battlefield awareness.', "Rare"),
    ('Dwarf', 'Stout warriors of immense durability, but slower on their feet.', "Rare"),
    ('Demon', 'Infernal beings whose raw power far exceeds their mortal counterparts.', "Legendary"),
    ('Angel', 'Celestial warriors with heightened resilience and intellect.', "Legendary"),
]

BUFFS = [
    ('Demon', 'Strength',   2),
    ('Demon', 'Durability', 1),
    ('Demon', 'Speed',     -1),
    ('Angel', 'Durability', 2),
    ('Angel', 'IQ',         1),
    ('Elf',   'Speed',      1),
    ('Elf',   'BIQ',        1),
    ('Elf',   'Strength',  -1),
    ('Dwarf', 'Durability', 3),
    ('Dwarf', 'Stamina',    1),
    ('Dwarf', 'Speed',     -2),
]

%sql --close sqlite:///characters.db

conn = sqlite3.connect('characters.db')
cur  = conn.cursor()

for name, description, rarity in SPECIES:
    cur.execute(
        "INSERT OR IGNORE INTO species (species, description, rarity) VALUES (?, ?, ?)",
        (name, description, rarity)
    )

for species_name, attribute, modifier in BUFFS:
    cur.execute(
        """INSERT OR IGNORE INTO species_buffs (species_id, attribute, level_modifier)
           VALUES ((SELECT id FROM species WHERE species = ?), ?, ?)""",
        (species_name, attribute, modifier)
    )

cur.execute(
    "SELECT * FROM species s JOIN species_buffs sb ON s.id = sb.species_id;"
)

print(cur.fetchall())

conn.commit()
conn.close()

[(4, 'Demon', 'Infernal beings whose raw power far exceeds their mortal counterparts.', 'Legendary', 4, 'Strength', 2), (4, 'Demon', 'Infernal beings whose raw power far exceeds their mortal counterparts.', 'Legendary', 4, 'Durability', 1), (4, 'Demon', 'Infernal beings whose raw power far exceeds their mortal counterparts.', 'Legendary', 4, 'Speed', -1), (5, 'Angel', 'Celestial warriors with heightened resilience and intellect.', 'Legendary', 5, 'Durability', 2), (5, 'Angel', 'Celestial warriors with heightened resilience and intellect.', 'Legendary', 5, 'IQ', 1), (2, 'Elf', 'Ancient race with unmatched speed and battlefield awareness.', 'Rare', 2, 'Speed', 1), (2, 'Elf', 'Ancient race with unmatched speed and battlefield awareness.', 'Rare', 2, 'BIQ', 1), (2, 'Elf', 'Ancient race with unmatched speed and battlefield awareness.', 'Rare', 2, 'Strength', -1), (3, 'Dwarf', 'Stout warriors of immense durability, but slower on their feet.', 'Rare', 3, 'Durability', 3), (3, 'Dwarf', 'Stout 